In [1]:
!pip install -q transformers sentencepiece x-transformers optuna scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.0/110.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.1 MB/s eta 0:00:00


In [2]:
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.44.2 tokenizers==0.19.1

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 115.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 94.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 76.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 48.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 94.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 15.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 35.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 14.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/19

In [3]:
import os
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from x_transformers import Decoder

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

import optuna

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [5]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE:", device)

DEVICE: cuda


In [6]:
MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"

MAX_LEN = 128

EPOCHS = 100

BATCH_SIZE = 64

NUM_LABELS = 2

In [7]:
# =========================================================
# CHECKPOINT DIRECTORY
# =========================================================
# สำหรับ Kaggle
CHECKPOINT_DIR = "/kaggle/working/checkpoints"

# สำหรับ Colab ให้ใช้บรรทัดนี้แทน:
# CHECKPOINT_DIR = "/content/checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

STUDY_DB_PATH = f"sqlite:///{CHECKPOINT_DIR}/optuna_study.db"
BEST_MODEL_PATH = f"{CHECKPOINT_DIR}/best_model.pt"
FINAL_MODEL_PATH = f"{CHECKPOINT_DIR}/final_depression_model.pt"

print("CHECKPOINT_DIR:", CHECKPOINT_DIR)


CHECKPOINT_DIR: /kaggle/working/checkpoints


In [8]:
TRAIN_PATH = "/kaggle/input/datasets/kmkimmy/thai-depression/Thai Depression/train.json"

VALID_PATH = "/kaggle/input/datasets/kmkimmy/thai-depression/Thai Depression/valid.json"

TEST_PATH = "/kaggle/input/datasets/kmkimmy/thai-depression/Thai Depression/test.json"

In [9]:
def load_json_dataset(path):

    with open(path, "r", encoding="utf-8") as f:

        raw_data = json.load(f)

    texts = []

    labels = []

    for item in raw_data:

        if len(item) != 2:
            continue

        text, label = item

        if not isinstance(text, str):
            continue

        # =================================================
        # CLEAN TEXT
        # =================================================
        text = text.replace(
            "---- “ ----",
            " "
        )

        text = text.replace(
            "\n",
            " "
        )

        text = " ".join(
            text.split()
        )

        text = text.strip()

        if len(text) < 2:
            continue

        # =================================================
        # LABEL
        # =================================================
        if label == "depression":

            label_id = 1

        elif label == "no_depression":

            label_id = 0

        else:
            continue

        texts.append(text)

        labels.append(label_id)

    df = pd.DataFrame({
        "text": texts,
        "label": labels
    })

    return df


In [10]:
train_df = load_json_dataset(TRAIN_PATH)

val_df = load_json_dataset(VALID_PATH)

test_df = load_json_dataset(TEST_PATH)

In [11]:
print("\nTRAIN:", len(train_df))

print("VALID:", len(val_df))

print("TEST :", len(test_df))

print("\nTRAIN LABEL COUNTS")
print(train_df["label"].value_counts())

print("\nVALID LABEL COUNTS")
print(val_df["label"].value_counts())

print("\nTEST LABEL COUNTS")
print(test_df["label"].value_counts())


TRAIN: 25004
VALID: 3335
TEST : 5004

TRAIN LABEL COUNTS
label
1    12765
0    12239
Name: count, dtype: int64

VALID LABEL COUNTS
label
1    1703
0    1632
Name: count, dtype: int64

TEST LABEL COUNTS
label
1    2556
0    2448
Name: count, dtype: int64


In [12]:
TEXT_COLUMN = "text"

LABEL_COLUMN = "label"

LABEL_NAMES = [
    "no_depression",
    "depression"
]

In [13]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

tokenizer_config.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/905k [00:00<?, ?B/s]

In [14]:
class EmotionDataset(Dataset):

    def __init__(self, df):

        self.texts = df[TEXT_COLUMN].tolist()

        self.labels = df[LABEL_COLUMN].tolist()

    def __len__(self):

        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])

        label = self.labels[idx]

        encoding = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return {

            "input_ids":
                encoding["input_ids"].squeeze(0),

            "attention_mask":
                encoding["attention_mask"].squeeze(0),

            "labels":
                torch.tensor(
                    label,
                    dtype=torch.long
                )
        }


In [15]:
train_dataset = EmotionDataset(train_df)

val_dataset = EmotionDataset(val_df)

test_dataset = EmotionDataset(test_df)

In [16]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [17]:
class_counts = (
    train_df[LABEL_COLUMN]
    .value_counts()
    .sort_index()
    .values
)

class_weights = 1.0 / class_counts

class_weights = (
    class_weights /
    class_weights.sum()
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
).to(device)

print("\nCLASS WEIGHTS:")
print(class_weights)



CLASS WEIGHTS:
tensor([0.5105, 0.4895], device='cuda:0')


In [18]:
class AttentionPooling(nn.Module):

    def __init__(self, hidden_size):

        super().__init__()

        self.attention = nn.Sequential(

            nn.Linear(
                hidden_size,
                hidden_size
            ),

            nn.Tanh(),

            nn.Linear(
                hidden_size,
                1
            )
        )

    def forward(self, x, mask):

        scores = self.attention(x).squeeze(-1)

        scores = scores.masked_fill(
            mask == 0,
            torch.finfo(scores.dtype).min
        )

        weights = torch.softmax(
            scores,
            dim=1
        )

        pooled = torch.sum(
            x * weights.unsqueeze(-1),
            dim=1
        )

        return pooled


In [19]:
class EmotionModel(nn.Module):

    def __init__(
        self,
        depth=1,
        heads=2,
        attn_dropout=0.2,
        ff_dropout=0.2,
        ff_mult=2
    ):

        super().__init__()

        self.encoder = AutoModel.from_pretrained(
            MODEL_NAME
        )

        hidden_size = (
            self.encoder
            .config
            .hidden_size
        )

        self.decoder = Decoder(

            dim=hidden_size,

            depth=depth,

            heads=heads,

            attn_dropout=attn_dropout,

            ff_dropout=ff_dropout,

            ff_mult=ff_mult
        )

        self.pooling = AttentionPooling(
            hidden_size
        )

        self.dropout = nn.Dropout(0.4)

        self.fc = nn.Linear(
            hidden_size,
            NUM_LABELS
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        x = outputs.last_hidden_state

        x = self.decoder(x)

        x = self.pooling(
            x,
            attention_mask
        )

        x = self.dropout(x)

        logits = self.fc(x)

        return logits

In [20]:
def evaluate(model, loader):

    model.eval()

    all_labels = []

    all_preds = []

    with torch.no_grad():

        for batch in loader:

            input_ids = (
                batch["input_ids"]
                .to(device)
            )

            attention_mask = (
                batch["attention_mask"]
                .to(device)
            )

            labels = (
                batch["labels"]
                .to(device)
            )

            logits = model(
                input_ids,
                attention_mask
            )

            preds = torch.argmax(
                logits,
                dim=1
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    weighted_f1 = f1_score(
        all_labels,
        all_preds,
        average="weighted"
    )

    return {

        "accuracy":
            accuracy,

        "macro_f1":
            macro_f1,

        "weighted_f1":
            weighted_f1,

        "labels":
            np.array(all_labels),

        "preds":
            np.array(all_preds)
    }

In [21]:
def train_model(
    model,
    train_loader,
    val_loader,
    lr,
    weight_decay,
    resume_checkpoint=None
):

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    total_steps = (
        len(train_loader)
        * EPOCHS
    )

    scheduler = (
        get_linear_schedule_with_warmup(

            optimizer,

            num_warmup_steps=int(
                0.1 * total_steps
            ),

            num_training_steps=total_steps
        )
    )

    scaler = torch.cuda.amp.GradScaler()

    best_macro_f1 = 0
    start_epoch = 0

    # =================================================
    # RESUME FROM CHECKPOINT (ถ้ามี)
    # =================================================
    if resume_checkpoint and os.path.exists(resume_checkpoint):

        print(f"\n🔄 กำลัง resume จาก: {resume_checkpoint}")

        ckpt = torch.load(
            resume_checkpoint,
            map_location=device
        )

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        scaler.load_state_dict(ckpt["scaler_state_dict"])
        start_epoch = ckpt["epoch"] + 1
        best_macro_f1 = ckpt.get("best_macro_f1", 0)

        print(f"✅ Resume สำเร็จ! เริ่มต่อจาก epoch {start_epoch + 1}")

    for epoch in range(start_epoch, EPOCHS):

        model.train()

        total_loss = 0

        loop = tqdm(train_loader)

        for batch in loop:

            input_ids = (
                batch["input_ids"]
                .to(device)
            )

            attention_mask = (
                batch["attention_mask"]
                .to(device)
            )

            labels = (
                batch["labels"]
                .to(device)
            )

            optimizer.zero_grad()

            with torch.cuda.amp.autocast():

                logits = model(
                    input_ids,
                    attention_mask
                )

                loss = criterion(
                    logits,
                    labels
                )

            scaler.scale(loss).backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            scaler.step(optimizer)

            scaler.update()

            scheduler.step()

            total_loss += loss.item()

            loop.set_description(
                f"Epoch {epoch+1}/{EPOCHS}"
            )

            loop.set_postfix(
                loss=loss.item()
            )

        val_result = evaluate(
            model,
            val_loader
        )

        print("\n======================")

        print(f"Epoch {epoch+1}")

        print("======================")

        print(
            f"VAL ACCURACY   : "
            f"{val_result['accuracy']:.4f}"
        )

        print(
            f"VAL MACRO F1   : "
            f"{val_result['macro_f1']:.4f}"
        )

        print(
            f"VAL WEIGHTED F1: "
            f"{val_result['weighted_f1']:.4f}"
        )

        # =================================================
        # 💾 SAVE CHECKPOINT (เก็บแค่อันล่าสุด)
        # =================================================
        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_macro_f1": best_macro_f1,
        }

        ckpt_path = f"{CHECKPOINT_DIR}/last_checkpoint.pt"

        torch.save(checkpoint, ckpt_path)

        print(f"💾 Checkpoint บันทึกที่: {ckpt_path}")

        # =================================================
        # ⭐ SAVE BEST MODEL
        # =================================================
        if (
            val_result["macro_f1"]
            > best_macro_f1
        ):

            best_macro_f1 = (
                val_result["macro_f1"]
            )

            torch.save(
                model.state_dict(),
                BEST_MODEL_PATH
            )

            print(
                f"Best model อัปเดต! "
                f"(F1: {best_macro_f1:.4f})"
            )

    model.load_state_dict(
        torch.load(
            BEST_MODEL_PATH,
            map_location=device
        )
    )

    return model


In [22]:
def objective(trial):

    depth = trial.suggest_int(
        "depth",
        1,
        2
    )

    heads = trial.suggest_categorical(
        "heads",
        [2, 4]
    )

    attn_dropout = trial.suggest_float(
        "attn_dropout",
        0.2,
        0.5
    )

    ff_dropout = trial.suggest_float(
        "ff_dropout",
        0.2,
        0.5
    )

    ff_mult = trial.suggest_int(
        "ff_mult",
        2,
        4
    )

    lr = trial.suggest_float(
        "lr",
        1e-5,
        2e-4,
        log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-4,
        1e-2,
        log=True
    )

    model = EmotionModel(

        depth=depth,

        heads=heads,

        attn_dropout=attn_dropout,

        ff_dropout=ff_dropout,

        ff_mult=ff_mult

    ).to(device)

    model = train_model(
        model,
        train_loader,
        val_loader,
        lr,
        weight_decay
    )

    val_result = evaluate(
        model,
        val_loader
    )

    return val_result["macro_f1"]

In [23]:
# ลบ checkpoint เก่าเพื่อเคลียร์พื้นที่
import glob

old_files = glob.glob(f"{CHECKPOINT_DIR}/checkpoint_epoch_*.pt")

for f in old_files:
    os.remove(f)
    print(f"🗑️ ลบ: {f}")

print(f"\nลบ {len(old_files)} ไฟล์เรียบร้อย")



ลบ 0 ไฟล์เรียบร้อย


In [24]:
import shutil

OLD_PATH = "/kaggle/input/datasets/kmkimmy/last-checkpoint2"

# copy ทุกไฟล์ที่ต้องการ
for filename in ["last_checkpoint.pt", "best_model.pt", "optuna_study.db"]:
    src = f"{OLD_PATH}/{filename}"
    dst = f"{CHECKPOINT_DIR}/{filename}"
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"✅ copy {filename} สำเร็จ!")
    else:
        print(f"⚠️ ไม่เจอ {filename} — ข้ามไป")


✅ copy last_checkpoint.pt สำเร็จ!
✅ copy best_model.pt สำเร็จ!
✅ copy optuna_study.db สำเร็จ!


In [25]:
study = optuna.create_study(
    direction="maximize",
    study_name="thai-depression-optim",
    storage=STUDY_DB_PATH,
    load_if_exists=True
)

print(f"📊 Trial เดิมที่มี: {len(study.trials)}")

MAX_TRIALS = 10

remaining = MAX_TRIALS - len(study.trials)

if remaining > 0:
    study.optimize(
        objective,
        n_trials=remaining,
        catch=(RuntimeError,)
    )
else:
    print("✅ ครบ 10 trial แล้ว ข้ามไป cell ถัดไป")

best_params = study.best_params
print("\nBEST PARAMS:", best_params)


[I 2026-05-30 15:12:49,621] Using an existing study with name 'thai-depression-optim' instead of creating a new one.


📊 Trial เดิมที่มี: 10
✅ ครบ 10 trial แล้ว ข้ามไป cell ถัดไป

BEST PARAMS: {'depth': 2, 'heads': 4, 'attn_dropout': 0.2240577833126539, 'ff_dropout': 0.4922163075549199, 'ff_mult': 4, 'lr': 3.0331847361552553e-05, 'weight_decay': 0.0002821544949248052}


In [26]:
final_model = EmotionModel(

    depth=best_params["depth"],

    heads=best_params["heads"],

    attn_dropout=best_params["attn_dropout"],

    ff_dropout=best_params["ff_dropout"],

    ff_mult=best_params["ff_mult"]

).to(device)

model.safetensors:   0%|          | 0.00/423M [00:00<?, ?B/s]

In [27]:
final_model = train_model(

    final_model,

    train_loader,

    val_loader,

    best_params["lr"],

    best_params["weight_decay"]
)

Epoch 1/100: 100%|██████████| 391/391 [05:18<00:00,  1.23it/s, loss=0.652]



Epoch 1
VAL ACCURACY   : 0.6567
VAL MACRO F1   : 0.6488
VAL WEIGHTED F1: 0.6477
💾 Checkpoint บันทึกที่: /kaggle/working/checkpoints/last_checkpoint.pt
Best model อัปเดต! (F1: 0.6488)


Epoch 2/100: 100%|██████████| 391/391 [05:18<00:00,  1.23it/s, loss=0.457]



Epoch 2
VAL ACCURACY   : 0.7718
VAL MACRO F1   : 0.7718
VAL WEIGHTED F1: 0.7718
💾 Checkpoint บันทึกที่: /kaggle/working/checkpoints/last_checkpoint.pt
Best model อัปเดต! (F1: 0.7718)


Epoch 3/100: 100%|██████████| 391/391 [05:18<00:00,  1.23it/s, loss=0.432]


...


Epoch 98
VAL ACCURACY   : 0.8375
VAL MACRO F1   : 0.8374
VAL WEIGHTED F1: 0.8375
💾 Checkpoint บันทึกที่: /kaggle/working/checkpoints/last_checkpoint.pt


Epoch 99/100: 100%|██████████| 391/391 [05:18<00:00,  1.23it/s, loss=1.7e-7]



Epoch 99
VAL ACCURACY   : 0.8375
VAL MACRO F1   : 0.8374
VAL WEIGHTED F1: 0.8375
💾 Checkpoint บันทึกที่: /kaggle/working/checkpoints/last_checkpoint.pt


Epoch 100/100: 100%|██████████| 391/391 [05:18<00:00,  1.23it/s, loss=1.7e-7]



Epoch 100
VAL ACCURACY   : 0.8375
VAL MACRO F1   : 0.8374
VAL WEIGHTED F1: 0.8375
💾 Checkpoint บันทึกที่: /kaggle/working/checkpoints/last_checkpoint.pt


In [28]:
test_result = evaluate(
    final_model,
    test_loader
)

print("\n======================")

print("FINAL RESULT")

print("======================")

print(
    f"Accuracy     : "
    f"{test_result['accuracy']:.4f}"
)

print(
    f"Macro F1     : "
    f"{test_result['macro_f1']:.4f}"
)

print(
    f"Weighted F1  : "
    f"{test_result['weighted_f1']:.4f}"
)


FINAL RESULT
Accuracy     : 0.8183
Macro F1     : 0.8182
Weighted F1  : 0.8183


In [29]:
print("\n======================")

print("CLASSIFICATION REPORT")

print("======================")

print(
    classification_report(

        test_result["labels"],

        test_result["preds"],

        target_names=LABEL_NAMES,

        zero_division=0
    )
)


CLASSIFICATION REPORT
               precision    recall  f1-score   support

no_depression       0.82      0.80      0.81      2448
   depression       0.82      0.83      0.82      2556

     accuracy                           0.82      5004
    macro avg       0.82      0.82      0.82      5004
 weighted avg       0.82      0.82      0.82      5004



In [30]:
print("\n======================")

print("CONFUSION MATRIX")

print("======================")

cm = confusion_matrix(
    test_result["labels"],
    test_result["preds"]
)

print(cm)


CONFUSION MATRIX
[[1969  479]
 [ 430 2126]]


In [31]:
SAVE_PATH = FINAL_MODEL_PATH

torch.save(
    {
        "model_state_dict":
            final_model.state_dict(),
        "best_params":
            best_params,
        "labels":
            LABEL_NAMES
    },
    SAVE_PATH
)

print("\nMODEL SAVED:", SAVE_PATH)



MODEL SAVED: /kaggle/working/checkpoints/final_depression_model.pt


In [32]:
# =========================================================
# LOAD MODEL กลับมาใช้งาน (สำหรับใช้ภายหลัง)
# =========================================================
saved = torch.load(
    FINAL_MODEL_PATH,
    map_location=device
)

print("SAVED PARAMS:", saved["best_params"])
print("LABELS:", saved["labels"])

loaded_params = saved["best_params"]

loaded_model = EmotionModel(

    depth=loaded_params["depth"],

    heads=loaded_params["heads"],

    attn_dropout=loaded_params["attn_dropout"],

    ff_dropout=loaded_params["ff_dropout"],

    ff_mult=loaded_params["ff_mult"]

).to(device)

loaded_model.load_state_dict(
    saved["model_state_dict"]
)

loaded_model.eval()

print("\nโหลดโมเดลสำเร็จ! พร้อมใช้งาน")


SAVED PARAMS: {'depth': 2, 'heads': 4, 'attn_dropout': 0.2240577833126539, 'ff_dropout': 0.4922163075549199, 'ff_mult': 4, 'lr': 3.0331847361552553e-05, 'weight_decay': 0.0002821544949248052}
LABELS: ['no_depression', 'depression']

โหลดโมเดลสำเร็จ! พร้อมใช้งาน
